# Primeiros passos com TabPFN

**Para quem está começando.** Nenhuma linha de código aqui assume que você já viu machine learning. A ideia é rodar tudo de cima para baixo, célula por célula, e olhar os números que aparecem.

O que vamos fazer, em ordem:

1. assistir a um vídeo curto sobre o assunto;
2. carregar duas tabelas de dados reais (uma de classificação, uma de regressão);
3. rodar um modelo tradicional (Random Forest) e um modelo pré-treinado (TabPFN);
4. comparar os resultados numa tabela.

Roda no Google Colab (CPU ou GPU) ou no Jupyter do seu computador. No Colab, é só apertar o play de cada célula na ordem.


**Vídeo:** [tutorial prático de TabPFN em Python](https://www.youtube.com/watch?v=8RbxKffv_BA)


## O que mudou nos dados em tabela

Você tem uma tabela: cada **linha** é um exemplo (um paciente), cada **coluna** é uma informação sobre ele (idade, medida de exame). Uma das colunas é o que você quer prever. Esse é o problema mais comum em dados do mundo real, e por vinte anos a resposta foi sempre a mesma: usar **árvores de decisão com boosting**, como Random Forest, XGBoost e LightGBM. Elas funcionam aprendendo regras do tipo "se esta medida for maior que tal valor, então...". São rápidas, lidam bem com colunas de escalas diferentes e ganham de redes neurais nesse tipo de dado.

O **TabPFN** faz diferente. Antes de você usá-lo, ele já foi treinado em milhões de tabelas inventadas por computador. Ele **não aprende com a sua tabela**: ele lê a sua tabela de treino como *contexto* e responde na hora, em uma passada. Isso se chama *in-context learning*.

O que isso muda no código? `fit` não treina nada (não existe descida de gradiente aqui), então é quase instantâneo. Já o `predict` fica caro: o modelo relê o contexto inteiro a cada previsão. E não há quase nada para ajustar: sem busca de hiperparâmetros, sem validação cruzada, sem feature engineering.

O que observar nos resultados: os números (ROC-AUC e F1 na classificação, RMSE, MAE e R² na regressão) e os **tempos**. Em tabelas pequenas, o esperado é o modelo pré-treinado empatar ou ganhar com pouco esforço. E lembre: uma única divisão de treino e teste mostra o mecanismo, não decide qual modelo é melhor no geral.


## Passo 0: instalar o que falta

No Colab (e num ambiente novo) esta linha baixa a biblioteca `tabpfn`, que já traz o PyTorch. Se você já instalou, ela não faz nada.


In [ ]:
%pip install tabpfn==9.0.0


In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion

# A semente fixa faz o sorteio ser sempre o mesmo: qualquer pessoa que rodar este
# notebook vai ver os mesmos números que você.
RANDOM_STATE = 42
TEST_SIZE = 0.25

print("✅ tudo importado")


# Parte 1: classificação

Vamos prever se um tumor é **maligno** ou **benigno** a partir de medidas do exame. É uma escolha entre duas opções, ou seja, classificação. A tabela vem pronta dentro do scikit-learn: 569 exames, 30 medidas.


In [ ]:
# Passo 1: carregar a tabela
data = load_breast_cancer(as_frame=True).frame

print("linhas e colunas:", data.shape)
print("a coluna que queremos prever é 'target':", data["target"].unique())


In [ ]:
# Passo 2: separar em treino e teste
# O modelo aprende com uma parte e é avaliado na outra, que ele nunca viu.
train, test = train_test_split(
    data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["target"]
)

X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]

print(f"treino: {X_train.shape[0]} exames | teste: {X_test.shape[0]} exames")


In [ ]:
# Passo 3 (modelo tradicional): Random Forest
# Guardamos os resultados de cada modelo nesta lista, para montar a tabela no final.
clf_results = []

model = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)

start = time.time()
model.fit(X_train, y_train)
fit_s = time.time() - start

start = time.time()
proba = model.predict_proba(X_test)[:, 1]
predict_s = time.time() - start

clf_results.append({
    "model": "Random Forest",
    "roc_auc": roc_auc_score(y_test, proba),
    "f1_macro": f1_score(y_test, (proba > 0.5).astype(int), average="macro"),
    "fit_s": round(fit_s, 2),
    "predict_s": round(predict_s, 2),
    "total_s": round(fit_s + predict_s, 2),
})
print(clf_results[-1])


In [ ]:
# Passo 4 (modelo pré-treinado): TabPFN
# Primeiro uso baixa os pesos (~100 MB). Fazemos uma rodada rápida em poucas linhas
# para o download ficar fora do cronômetro.
speed_test = TabPFNClassifier.create_default_for_version(ModelVersion.V2, random_state=RANDOM_STATE)
speed_test.fit(X_train[:100], y_train[:100])

# random_state fixa a semente: os números saem iguais aos dos outros notebooks da série.
model = TabPFNClassifier.create_default_for_version(ModelVersion.V2, random_state=RANDOM_STATE)

start = time.time()
model.fit(X_train, y_train)
fit_s = time.time() - start

start = time.time()
proba = model.predict_proba(X_test)[:, 1]
predict_s = time.time() - start

clf_results.append({
    "model": "TabPFN v2",
    "roc_auc": roc_auc_score(y_test, proba),
    "f1_macro": f1_score(y_test, (proba > 0.5).astype(int), average="macro"),
    "fit_s": round(fit_s, 2),
    "predict_s": round(predict_s, 2),
    "total_s": round(fit_s + predict_s, 2),
})
print(clf_results[-1])


### Como ler isso

`roc_auc` mede se o modelo **ordena** bem os exemplos (1,0 seria perfeito, 0,5 seria chute). `f1_macro` mistura acerto e erro nas duas classes sem favorecer a mais comum. E repare na coluna `predict_s`: é ali que o modelo pré-treinado cobra o preço, porque ele relê o contexto a cada previsão.


In [ ]:
# Passo 5: a tabela comparativa da classificação
clf_table = pd.DataFrame(clf_results).set_index("model").sort_values("roc_auc", ascending=False)
clf_table.round(4)


# Parte 2: regressão

Agora o alvo é um **número**, não uma categoria: prever o preço mediano de casas na Califórnia. A tabela original tem 20 640 linhas; usamos 2 500 para o notebook rodar rápido.


In [ ]:
# Passo 6: carregar a tabela (mesma semente, para dar sempre o mesmo sorteio)
housing = fetch_california_housing(as_frame=True).frame
data = housing.rename(columns={"MedHouseVal": "target"}).sample(n=2500, random_state=RANDOM_STATE)

print("linhas e colunas:", data.shape)


In [ ]:
# Passo 7: separar em treino e teste
train, test = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)

X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]

print(f"treino: {X_train.shape[0]} casas | teste: {X_test.shape[0]} casas")

reg_results = []


In [ ]:
# Passo 8 (modelo tradicional): Random Forest
model = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)

start = time.time()
model.fit(X_train, y_train)
fit_s = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
predict_s = time.time() - start

reg_results.append({
    "model": "Random Forest",
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "r2": float(r2_score(y_test, y_pred)),
    "fit_s": round(fit_s, 2),
    "predict_s": round(predict_s, 2),
    "total_s": round(fit_s + predict_s, 2),
})
print(reg_results[-1])


In [ ]:
# Passo 9 (modelo pré-treinado): TabPFN
# O TabPFN recusa mais de 1000 linhas de treino quando roda na CPU (é lento, não é erro):
# aqui temos 1875, então liberamos o limite. Em GPU essa opção não muda nada.
model = TabPFNRegressor.create_default_for_version(
    ModelVersion.V2, random_state=RANDOM_STATE, ignore_pretraining_limits=True
)

start = time.time()
model.fit(X_train, y_train)
fit_s = time.time() - start

start = time.time()
y_pred = model.predict(X_test)
predict_s = time.time() - start

reg_results.append({
    "model": "TabPFN v2",
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "r2": float(r2_score(y_test, y_pred)),
    "fit_s": round(fit_s, 2),
    "predict_s": round(predict_s, 2),
    "total_s": round(fit_s + predict_s, 2),
})
print(reg_results[-1])


### Como ler isso

`rmse` e `mae` são erros: quanto **menor**, melhor (estão na mesma unidade do alvo, aqui em centenas de milhares de dólares). `r2` é a fração da variação explicada: quanto **maior**, melhor. Compare os dois modelos e, principalmente, os tempos: o TabPFN praticamente não gasta tempo no `fit` e gasta bastante no `predict`.


In [ ]:
# Passo 10: a tabela comparativa da regressão
reg_table = pd.DataFrame(reg_results).set_index("model").sort_values("rmse")
reg_table.round(4)


## Resumo

O TabPFN não foi treinado na sua tabela: ele leu o seu treino como contexto. Isso derruba o trabalho de ajuste (nada de busca de hiperparâmetros) e transfere o custo para a previsão. Em tabelas pequenas, como estas, ele costuma empatar ou ganhar do modelo tradicional.

Dois cuidados que valem para tudo neste notebook: **uma divisão só de treino/teste não decide campeonato** (para afirmar algo, repita com sementes diferentes e reporte média e desvio) e **tempo de CPU não é tempo de GPU** (no Colab, rode com GPU para ver a diferença no `predict_s`).

### Referências

**TabPFN**
Hollmann, N., Müller, S., Purucker, L., Krishnakumar, A., Körfer, M., Hoo, S. B., Schirrmeister, R. T., & Hutter, F. (2025). *Accurate predictions on small data with a tabular foundation model.* **Nature** 637, 319–326. DOI: 10.1038/s41586-024-08328-6. Preprint: arXiv:2402.12884.

**Por que árvores dominavam dados em tabela**
Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS 2022, Datasets and Benchmarks Track. arXiv:2207.08815.

**O modelo da próxima parte**
Zhang, X., Maddix, D. C., Yin, J., Erickson, N., Ansari, A. F., Han, B., Zhang, S., Akoglu, L., Faloutsos, C., Mahoney, M. W., Hu, C., Rangwala, H., Karypis, G., & Wang, B. (2025). *Mitra: Mixed Synthetic Priors for Enhancing Tabular Foundation Models.* NeurIPS 2025. arXiv:2510.21204.

**Código e documentação**
https://github.com/PriorLabs/TabPFN · https://docs.priorlabs.ai

```bibtex
@article{hollmann2025tabpfn,
  title   = {Accurate predictions on small data with a tabular foundation model},
  author  = {Hollmann, Noah and M{\"u}ller, Samuel and Purucker, Lennart and
             Krishnakumar, Arjun and K{\"o}rfer, Max and Hoo, Shi Bin and
             Schirrmeister, Robin Tibor and Hutter, Frank},
  journal = {Nature},
  volume  = {637},
  pages   = {319--326},
  year    = {2025},
  doi     = {10.1038/s41586-024-08328-6}
}

@inproceedings{grinsztajn2022trees,
  title     = {Why do tree-based models still outperform deep learning on typical tabular data?},
  author    = {Grinsztajn, L{\'e}o and Oyallon, Edouard and Varoquaux, Ga{\"e}l},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS), Datasets and Benchmarks Track},
  year      = {2022},
  eprint    = {2207.08815},
  archivePrefix = {arXiv}
}

@inproceedings{zhang2025mitra,
  title     = {Mitra: Mixed Synthetic Priors for Enhancing Tabular Foundation Models},
  author    = {Zhang, Xiyuan and Maddix, Danielle C. and Yin, Junming and Erickson, Nick and
               Ansari, Abdul Fatir and Han, Boran and Zhang, Shuai and Akoglu, Leman and
               Faloutsos, Christos and Mahoney, Michael W. and Hu, Cuixiong and
               Rangwala, Huzefa and Karypis, George and Wang, Bernie},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS)},
  year      = {2025},
  eprint    = {2510.21204},
  archivePrefix = {arXiv}
}
```

**Próximo:** `02_mitra.ipynb`, o mesmo experimento com outro modelo pré-treinado.
